In [1]:
import os
import torch

# Print the NVIDIA CUDA visible devices environment variable
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

# Check if CUDA is available
cuda_available = torch.cuda.is_available()
print("Is CUDA available?:", cuda_available)

if cuda_available:
    print("GPU Device Count:", torch.cuda.device_count())
    print("Primary GPU Name:", torch.cuda.get_device_name(0))
else:
    print("No NVIDIA GPU detected by PyTorch.")


CUDA_VISIBLE_DEVICES: None
Is CUDA available?: True
GPU Device Count: 1
Primary GPU Name: AMD Radeon RX 7800 XT


In [2]:
import torch
import torch.nn.functional as F

In [3]:
# T = 5 tokens, d_model = 3
X = torch.tensor([[1., 0., 1.],   # The
                  [0., 2., 1.],   # tall
                  [1., 1., 0.],   # man
                  [2., 0., 1.],   # runs
                  [0., 1., 2.]])  # .

In [4]:
print(X)

tensor([[1., 0., 1.],
        [0., 2., 1.],
        [1., 1., 0.],
        [2., 0., 1.],
        [0., 1., 2.]])


In [5]:
# Weights as in × out, like the paper (d_k = 2, d_v = 4)
W_Q = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])                          # 3 × 2
print(f"W_Q:")
print(W_Q)
W_K = torch.tensor([[0., 1.], [1., 0.], [1., 0.]])                          # 3 × 2
print(f"W_K:")
print(W_K)
W_V = torch.tensor([[1., 0., 2., 0.], [0., 1., 0., 1.], [1., 0., 0., 2.]])  # 3 × 4
print(f"W_V:")
print(W_V)

W_Q:
tensor([[1., 0.],
        [0., 1.],
        [1., 1.]])
W_K:
tensor([[0., 1.],
        [1., 0.],
        [1., 0.]])
W_V:
tensor([[1., 0., 2., 0.],
        [0., 1., 0., 1.],
        [1., 0., 0., 2.]])


In [6]:
Q = X @ W_Q       # 5 x 2 - query and key vectors are 2 dimensional.
K = X @ W_K       # 5 x 2 - I did this so that all numbers like T = 5, d_k = 2, d_v = 4 d_model = 3 are all different and easy to see.
V = X @ W_V       # 5 x 4 - value vector is 4 dimensional, v1, v2, v3, v4

In [7]:
T, d_k = Q.shape                # T =5, d_k = 2

In [8]:
Attention_matrix = Q @ K.T
print(Attention_matrix.shape)
print(Attention_matrix)

torch.Size([5, 5])
tensor([[3., 6., 3., 4., 6.],
        [4., 3., 4., 7., 3.],
        [2., 3., 2., 3., 3.],
        [4., 9., 4., 5., 9.],
        [5., 6., 5., 8., 6.]])


In [9]:
# Scale attention matrix by len of query vector. I don't know why this is done
# But I will do it now regardless.
# BTW, len of query vector is same as len of key vector. => d_q = d_k
# In this case the query and the key vectors emitted are of length 2
# So d_k = 2

Attention_scores = Attention_matrix / (d_k ** 0.5)

In [10]:
print(Attention_scores.shape)

#Boilerplate to print numbers with 1 decimal places temporarily
old_precision = torch._tensor_str.PRINT_OPTS.precision # Save old precision
torch.set_printoptions(precision=1)                    # Temporarily modify it

print(Attention_scores)

torch.set_printoptions(precision=old_precision)       # Change it back

torch.Size([5, 5])
tensor([[2.1, 4.2, 2.1, 2.8, 4.2],
        [2.8, 2.1, 2.8, 4.9, 2.1],
        [1.4, 2.1, 1.4, 2.1, 2.1],
        [2.8, 6.4, 2.8, 3.5, 6.4],
        [3.5, 4.2, 3.5, 5.7, 4.2]])


In [11]:
allowed = torch.tril(torch.ones(T, T, dtype=torch.bool)) # T was 5
print(allowed)

tensor([[ True, False, False, False, False],
        [ True,  True, False, False, False],
        [ True,  True,  True, False, False],
        [ True,  True,  True,  True, False],
        [ True,  True,  True,  True,  True]])


torch.tril(...) keeps the lower triangle, including the diagonal, and zeros the rest. So allowed[t, s] is True when $s \le t$: token $t$ may look at itself and everything before it.

In [12]:
Attention_scores = Attention_scores.masked_fill(~allowed, float('-inf'))
# Seems like this does a msked fill on attention_scores based on the mask called "allowed". In allowed the top half was blacked out.
# So here we take the opposite of that "~allowed". That inverts the mask. Now we have selected the top right part of the mask as True. Then we put -inf on that newly selected part which is the top-right.
# This ultimately means that we have created a mask of -inf for the top right triangle.
# This is called the causal mask.
# 2. causal mask: True on and below the diagonal = "allowed to look"
print(Attention_scores)

tensor([[2.1213,   -inf,   -inf,   -inf,   -inf],
        [2.8284, 2.1213,   -inf,   -inf,   -inf],
        [1.4142, 2.1213, 1.4142,   -inf,   -inf],
        [2.8284, 6.3640, 2.8284, 3.5355,   -inf],
        [3.5355, 4.2426, 3.5355, 5.6569, 4.2426]])


In [13]:
# Now we can take the softmax
A = F.softmax(Attention_scores, dim = 1)
print(A)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6698, 0.3302, 0.0000, 0.0000, 0.0000],
        [0.2483, 0.5035, 0.2483, 0.0000, 0.0000],
        [0.0261, 0.8949, 0.0261, 0.0529, 0.0000],
        [0.0695, 0.1409, 0.0695, 0.5794, 0.1409]])


In [ ]:
# 4. Take A.V

Z = A @ V       # 5x4
print(Z)        # Z is the output of the attention layer,
# Actually Z is the output of one slef-Attention head. Several outputs from many such heads are combined and formed into the actual output of the attention layer. This is followed up by a MLP in the Transformer Block.

tensor([[2.0000, 0.0000, 2.0000, 2.0000],
        [1.6698, 0.6605, 1.3395, 2.6605],
        [1.2483, 1.2552, 0.9930, 2.7587],
        [1.1319, 1.8160, 0.3159, 3.7638],
        [2.3691, 0.4920, 2.5953, 2.6348]])
